# BioListen VN — Baseline Multi-task Model Training [Kaggle GPU Environment]

Notebook này thực hiện huấn luyện mô hình **Multi-task baseline** (Backbone: EfficientNet-B0) trên hạ tầng GPU của Kaggle.

### 💡 Tối ưu hóa đĩa cứng cho Kaggle (In-Memory ZIP-Reading):
Kaggle giới hạn không gian làm việc `/kaggle/working` là **20 GB**. Bộ dữ liệu sau khi giải nén chứa hàng ngàn tệp tin `.pt` nhỏ sẽ chiếm rất nhiều dung lượng và làm chậm I/O của hệ thống. 
- **Giải pháp:** Sử dụng lớp Dataset tùy chỉnh đọc trực tiếp các tệp `.pt` đặc trưng từ các tệp ZIP đặt tại `/kaggle/input/` vào RAM bằng `io.BytesIO` và `zipfile`.
- **Kết quả:** Tiết kiệm **100% dung lượng đĩa ảo** (0 MB ghi ra `/kaggle/working` cho dữ liệu), tốc độ nạp dữ liệu cực nhanh và tận dụng tối đa GPU (T4 hoặc P100) của Kaggle.

## 1. Hướng dẫn chuẩn bị dữ liệu trên Kaggle

Trước khi chạy notebook này, bạn cần thực hiện:
1. Tạo một Dataset mới trên Kaggle (ví dụ đặt tên là `biolisten-processed-dataset`).
2. Tải lên 6 tệp tin sau từ Google Drive của bạn:
   * `fsc22_processed.zip` & `fsc22_processed_metadata.csv`
   * `rfcx_tp_processed.zip` & `rfcx_tp_processed_metadata.csv`
   * `rfcx_fp_processed.zip` & `rfcx_fp_processed_metadata.csv`
3. Thêm Dataset vừa tạo vào Notebook này thông qua menu bên phải của Kaggle (`+ Add Input` -> `Search for your dataset`).

## 2. Kiểm tra Môi trường GPU & Cấu hình Đường dẫn

In [ ]:
import os
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Thiết bị huấn luyện: {device}")
if torch.cuda.is_available():
    print(f"  - Tên GPU: {torch.cuda.get_device_name(0)}")

# Cấu hình đường dẫn dataset đầu vào trên Kaggle
# Thay đổi tên thư mục 'biolisten-processed-dataset' tương ứng với tên Dataset bạn đặt trên Kaggle
KAGGLE_DATASET_NAME = 'biolisten-processed-dataset'
input_dir = f'/kaggle/input/{KAGGLE_DATASET_NAME}'

# Các file ZIP dữ liệu
fsc22_zip = os.path.join(input_dir, 'fsc22_processed.zip')
rfcx_tp_zip = os.path.join(input_dir, 'rfcx_tp_processed.zip')
rfcx_fp_zip = os.path.join(input_dir, 'rfcx_fp_processed.zip')

# Các file metadata CSV
fsc22_csv = os.path.join(input_dir, 'fsc22_processed_metadata.csv')
rfcx_tp_csv = os.path.join(input_dir, 'rfcx_tp_processed_metadata.csv')
rfcx_fp_csv = os.path.join(input_dir, 'rfcx_fp_processed_metadata.csv')

print("Tập tin đầu vào có sẵn:")
print("  - fsc22_zip:", os.path.exists(fsc22_zip))
print("  - rfcx_tp_zip:", os.path.exists(rfcx_tp_zip))
print("  - rfcx_fp_zip:", os.path.exists(rfcx_fp_zip))
print("  - fsc22_csv:", os.path.exists(fsc22_csv))

## 3. Khởi tạo In-Memory Dataset đọc trực tiếp từ ZIP

In [ ]:
import zipfile
import io
import pandas as pd
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

THREAT_CLASSES = [
    'Fire', 'Chainsaw', 'Handsaw', 'Helicopter', 'VehicleEngine', 
    'Axe', 'Gunshot', 'Footsteps'
]
threat_to_idx = {name: idx for idx, name in enumerate(THREAT_CLASSES)}
BACKGROUND_CLASS_IDX = 8

class BioListenKaggleDataset(Dataset):
    def __init__(self, samples_list, fsc22_zip, rfcx_tp_zip, rfcx_fp_zip):
        self.samples = samples_list
        self.fsc22_zip = fsc22_zip
        self.rfcx_tp_zip = rfcx_tp_zip
        self.rfcx_fp_zip = rfcx_fp_zip

        # Lập chỉ mục thành viên trong từng zip để tối ưu hóa truy xuất động
        self.fsc22_members = self._index_zip(fsc22_zip)
        self.rfcx_tp_members = self._index_zip(rfcx_tp_zip)
        self.rfcx_fp_members = self._index_zip(rfcx_fp_zip)

    def _index_zip(self, zip_path):
        if os.path.exists(zip_path):
            with zipfile.ZipFile(zip_path, 'r') as z:
                return set(z.namelist())
        return set()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        task_type = sample['task_type']
        dataset_name = sample['dataset_name']
        pt_name = sample['processed_pt_filename']
        
        if task_type == 'human':
            # Đọc nhị phân tensor từ ZIP của FSC22 trên RAM
            with zipfile.ZipFile(self.fsc22_zip, 'r') as z:
                with z.open(pt_name) as f:
                    tensor = torch.load(io.BytesIO(f.read()), map_location='cpu')
            
            category = sample['Class Name']
            threat_label = threat_to_idx.get(category, BACKGROUND_CLASS_IDX)
            species_label = torch.zeros(24)
            
        elif task_type == 'species':
            # Đọc nhị phân tensor từ ZIP của RFCx tương ứng
            is_fp = sample.get('is_fp', False)
            target_zip = self.rfcx_fp_zip if is_fp else self.rfcx_tp_zip
            
            with zipfile.ZipFile(target_zip, 'r') as z:
                with z.open(pt_name) as f:
                    tensor = torch.load(io.BytesIO(f.read()), map_location='cpu')
            
            species_label = torch.zeros(24)
            if not is_fp:
                species_id = int(sample['species_id'])
                species_label[species_id] = 1.0
            threat_label = BACKGROUND_CLASS_IDX
            
        return tensor, species_label, threat_label, task_type

all_train_samples = []
all_val_samples = []

# Nạp FSC22
if os.path.exists(fsc22_csv):
    df_fsc = pd.read_csv(fsc22_csv)
    df_fsc_train, df_fsc_val = train_test_split(df_fsc, test_size=0.2, random_state=42)
    for _, row in df_fsc_train.iterrows():
        d = row.to_dict()
        d['task_type'] = 'human'
        d['dataset_name'] = 'fsc22'
        all_train_samples.append(d)
    for _, row in df_fsc_val.iterrows():
        d = row.to_dict()
        d['task_type'] = 'human'
        d['dataset_name'] = 'fsc22'
        all_val_samples.append(d)
    print(f"FSC22: Train={len(df_fsc_train)} | Val={len(df_fsc_val)}")

# Nạp RFCx TP
if os.path.exists(rfcx_tp_csv):
    df_tp = pd.read_csv(rfcx_tp_csv)
    df_tp_train, df_tp_val = train_test_split(df_tp, test_size=0.2, random_state=42)
    for _, row in df_tp_train.iterrows():
        d = row.to_dict()
        d['task_type'] = 'species'
        d['dataset_name'] = 'rfcx_tp'
        d['is_fp'] = False
        all_train_samples.append(d)
    for _, row in df_tp_val.iterrows():
        d = row.to_dict()
        d['task_type'] = 'species'
        d['dataset_name'] = 'rfcx_tp'
        d['is_fp'] = False
        all_val_samples.append(d)
    print(f"RFCx TP: Train={len(df_tp_train)} | Val={len(df_tp_val)}")

# Nạp RFCx FP
if os.path.exists(rfcx_fp_csv):
    df_fp = pd.read_csv(rfcx_fp_csv)
    df_fp_train, df_fp_val = train_test_split(df_fp, test_size=0.2, random_state=42)
    for _, row in df_fp_train.iterrows():
        d = row.to_dict()
        d['task_type'] = 'species'
        d['dataset_name'] = 'rfcx_fp'
        d['is_fp'] = True
        all_train_samples.append(d)
    for _, row in df_fp_val.iterrows():
        d = row.to_dict()
        d['task_type'] = 'species'
        d['dataset_name'] = 'rfcx_fp'
        d['is_fp'] = True
        all_val_samples.append(d)
    print(f"RFCx FP: Train={len(df_fp_train)} | Val={len(df_fp_val)}")

# Khởi tạo Dataset & DataLoader
train_dataset = BioListenKaggleDataset(all_train_samples, fsc22_zip, rfcx_tp_zip, rfcx_fp_zip)
val_dataset = BioListenKaggleDataset(all_val_samples, fsc22_zip, rfcx_tp_zip, rfcx_fp_zip)

# Dùng num_workers=2 để tối ưu hóa nạp dữ liệu song song trên Kaggle
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Kaggle DataLoader ready! Train Batches={len(train_loader)} | Val Batches={len(val_loader)}")

## 4. Kiến trúc Mô hình Multi-task (Baseline: EfficientNet-B0)

In [ ]:
import torchvision.models as models

class BioListenMultiTaskModel(nn.Module):
    def __init__(self, num_species=24, num_threats=9, pretrained=True):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.efficientnet_b0(weights=weights)
        
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        
        self.species_head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_species)
        )
        self.human_head = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_threats)
        )

    def forward(self, x):
        features = self.backbone(x)
        species_logits = self.species_head(features)
        threat_logits = self.human_head(features)
        return species_logits, threat_logits

model = BioListenMultiTaskModel(pretrained=True).to(device)
print(f"Mô hình đã được dựng và chuyển vào: {device}")

## 5. Cấu hình Hàm Loss thích ứng & Optimizer

In [ ]:
def compute_multitask_loss(species_logits, threat_logits, species_targets, threat_targets, task_types, alpha=1.0, beta=1.0):
    is_species = torch.tensor([t == 'species' for t in task_types], device=species_logits.device)
    is_human = torch.tensor([t == 'human' for t in task_types], device=threat_logits.device)
    
    loss_species = torch.tensor(0.0, device=species_logits.device)
    loss_human = torch.tensor(0.0, device=threat_logits.device)
    
    if is_species.any():
        spec_logits_masked = species_logits[is_species]
        spec_targets_masked = species_targets[is_species].float()
        loss_species = nn.BCEWithLogitsLoss()(spec_logits_masked, spec_targets_masked)
        
    if is_human.any():
        threat_logits_masked = threat_logits[is_human]
        threat_targets_masked = threat_targets[is_human].long()
        loss_human = nn.CrossEntropyLoss()(threat_logits_masked, threat_targets_masked)
        
    total_loss = alpha * loss_species + beta * loss_human
    return total_loss, loss_species, loss_human

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
print("Optimizer AdamW và Cosine Annealing đã cấu hình xong.")

## 6. Vòng lặp Huấn luyện & Đồ thị Loss Curves

In [ ]:
# Lưu model tại thư mục output làm việc của Kaggle
base_models_dir = '/kaggle/working/models'
os.makedirs(base_models_dir, exist_ok=True)

existing_phases = [d for d in os.listdir(base_models_dir) if os.path.isdir(os.path.join(base_models_dir, d)) and d.startswith('phase_')]
phase_nums = []
for p in existing_phases:
    try:
        num = int(p.split('_')[1])
        phase_nums.append(num)
    except ValueError:
        pass

current_phase_num = max(phase_nums) + 1 if phase_nums else 1
current_phase_folder = f"phase_{current_phase_num:02d}"
current_save_dir = os.path.join(base_models_dir, current_phase_folder)
os.makedirs(current_save_dir, exist_ok=True)
print(f"Thư mục lưu mô hình: {current_save_dir}")

class EarlyStopping:
    def __init__(self, patience=7, delta=1e-4, save_dir=current_save_dir):
        self.patience = patience
        self.delta = delta
        self.save_dir = save_dir
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.save_count = 0

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            print(f"EarlyStopping Counter: {self.counter} / {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        os.makedirs(self.save_dir, exist_ok=True)
        self.save_count += 1
        
        latest_path = os.path.join(self.save_dir, 'best_model.pt')
        torch.save(model.state_dict(), latest_path)
        
        indexed_path = os.path.join(self.save_dir, f'best_model_{self.save_count}.pt')
        torch.save(model.state_dict(), indexed_path)
        
        print(f"[CHECKPOINT] Đã lưu mô hình tốt nhất (Lần {self.save_count}): {latest_path}")

from tqdm.notebook import tqdm

def train_epoch(model, loader, optimizer, device):
    model.train()
    epoch_loss = 0.0
    spec_losses = 0.0
    human_losses = 0.0
    
    pbar = tqdm(loader, desc='Training Batch', leave=True)
    for tensors, species_labels, threat_labels, task_types in pbar:
        tensors = tensors.to(device)
        species_labels = species_labels.to(device)
        threat_labels = threat_labels.to(device)
        
        optimizer.zero_grad()
        spec_logits, threat_logits = model(tensors)
        loss, l_spec, l_human = compute_multitask_loss(
            spec_logits, threat_logits, species_labels, threat_labels, task_types
        )
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        spec_losses += l_spec.item()
        human_losses += l_human.item()
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'spec_loss': f'{l_spec.item():.4f}',
            'human_loss': f'{l_human.item():.4f}'
        })
        
    num_batches = len(loader)
    return epoch_loss / num_batches, spec_losses / num_batches, human_losses / num_batches

def validate(model, loader, device):
    model.eval()
    val_loss = 0.0
    spec_losses = 0.0
    human_losses = 0.0
    
    pbar = tqdm(loader, desc='Validation Batch', leave=True)
    with torch.no_grad():
        for tensors, species_labels, threat_labels, task_types in pbar:
            tensors = tensors.to(device)
            species_labels = species_labels.to(device)
            threat_labels = threat_labels.to(device)
            
            spec_logits, threat_logits = model(tensors)
            loss, l_spec, l_human = compute_multitask_loss(
                spec_logits, threat_logits, species_labels, threat_labels, task_types
            )
            
            val_loss += loss.item()
            spec_losses += l_spec.item()
            human_losses += l_human.item()
            
            pbar.set_postfix({'val_loss': f'{loss.item():.4f}'})
            
    num_batches = len(loader)
    return val_loss / num_batches, spec_losses / num_batches, human_losses / num_batches

EPOCHS = 5
early_stopping = EarlyStopping(patience=3)

history = {
    'train_loss': [], 'train_spec_loss': [], 'train_human_loss': [],
    'val_loss': [], 'val_spec_loss': [], 'val_human_loss': []
}

print("Bắt đầu huấn luyện...")
for epoch in range(EPOCHS):
    train_loss, tr_spec, tr_human = train_epoch(model, train_loader, optimizer, device)
    val_loss, val_spec, val_human = validate(model, val_loader, device)
    
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_spec_loss'].append(tr_spec)
    history['train_human_loss'].append(tr_human)
    history['val_loss'].append(val_loss)
    history['val_spec_loss'].append(val_spec)
    history['val_human_loss'].append(val_human)
    
    # Sử dụng format chuẩn tránh ký tự thoát đặc biệt
    print(f'\n==> Epoch {epoch+1:02d}/{EPOCHS:02d} | '
          f'Train Loss: {train_loss:.4f} (Spec: {tr_spec:.4f}, Human: {tr_human:.4f}) | '
          f'Val Loss: {val_loss:.4f} (Spec: {val_spec:.4f}, Human: {val_human:.4f})')
    
    early_stopping(val_loss, model)
    if early_stopping.early_stop:
        print("Kích hoạt Early Stopping! Huấn luyện dừng sớm.")
        break

In [ ]:
# Vẽ đồ thị Loss curves
import matplotlib.pyplot as plt
import seaborn as sns

epochs_range = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
plt.plot(epochs_range, history['train_loss'], label='Train Loss', marker='o', color='royalblue')
plt.plot(epochs_range, history['val_loss'], label='Val Loss', marker='s', color='orange')
plt.title('Total Multi-task Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(epochs_range, history['train_spec_loss'], label='Train Species Loss', marker='o', color='forestgreen')
plt.plot(epochs_range, history['val_spec_loss'], label='Val Species Loss', marker='s', color='limegreen')
plt.title('Species Head Loss (BCE)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(epochs_range, history['train_human_loss'], label='Train Human Loss', marker='o', color='crimson')
plt.plot(epochs_range, history['val_human_loss'], label='Val Human Loss', marker='s', color='hotpink')
plt.title('Human Head Loss (CrossEntropy)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()